<a href="https://colab.research.google.com/github/sintojoy/thinkpalm-agentai-SintoJoy-ReAct_Agent/blob/main/src/Sinto_Joy_Lab_1_ReAct_Agent_QuizMaster_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Minimal ReAct Agent

import math

# -------------------------
# TOOL
# -------------------------
def dictionary_meaning_tool(word):
    """
    Simple dictionary meaning tool.
    """
    # This is a placeholder. In a real scenario, this would call an external dictionary API.
    meanings = {
        "hello": "A greeting used when meeting or acknowledging someone.",
        "world": "The earth, together with all of its countries, peoples, and natural features.",
        "python": "A large heavy-bodied nonvenomous constrictor snake; also, a high-level, interpreted programming language."
    }
    return meanings.get(word.lower(), f"Meaning for '{word}' not found.")


# -------------------------
# REACT AGENT
# -------------------------
def react_agent(user_query):

    print(f"\nUser Query: {user_query}\n")

    # Step 1: Reasoning
    print("Thought: I should find the meaning of the word using the dictionary meaning tool.")

    # Step 2: Tool Call
    # Extract the word from the query, assuming queries like "What is the meaning of 'word'?"
    # Or, if the query is just the word, assume that.
    word = user_query.replace("What is the meaning of", "").replace("\"", "").replace("'", "").replace("the word", "").replace("?", "").strip()

    print(f"Action: dictionary_meaning_tool('{word}')")

    observation = dictionary_meaning_tool(word)

    print(f"Observation: {observation}")

    # Step 3: Final Answer
    print("\nFinal Answer:", observation)


# -------------------------
# RUN AGENT
# -------------------------
query = "What is the meaning of 'python'?"

react_agent(query)


User Query: What is the meaning of 'python'?

Thought: I should find the meaning of the word using the dictionary meaning tool.
Action: dictionary_meaning_tool('python')
Observation: A large heavy-bodied nonvenomous constrictor snake; also, a high-level, interpreted programming language.

Final Answer: A large heavy-bodied nonvenomous constrictor snake; also, a high-level, interpreted programming language.


### Interactive Dropdown with Dictionary Lookup

We can use `ipywidgets` to create interactive elements like dropdowns. Here's an example of how a dropdown can be used to select a 'question' from a dictionary, and then display information related to that question.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Define a dictionary of questions and their possible answers or associated data
quiz_data = {
    "Question 1: What is the capital of France?": {
        "options": ["Berlin", "Paris", "Madrid", "Rome"],
        "answer": "Paris"
    },
    "Question 2: Which planet is known as the Red Planet?": {
        "options": ["Earth", "Mars", "Jupiter", "Venus"],
        "answer": "Mars"
    },
    "Question 3: Who painted the Mona Lisa?": {
        "options": ["Vincent van Gogh", "Pablo Picasso", "Leonardo da Vinci", "Claude Monet"],
        "answer": "Leonardo da Vinci"
    },
    "Question 4: What is the chemical symbol for water?": {
        "options": ["O2", "H2O", "CO2", "N2"],
        "answer": "H2O"
    },
    "Question 5: What is the largest ocean on Earth?": {
        "options": ["Atlantic", "Indian", "Arctic", "Pacific"],
        "answer": "Pacific"
    }
}

# List of question keys to maintain order
question_keys = list(quiz_data.keys())

# Create a Next button
next_button = widgets.Button(
    description='Next Question',
    disabled=True, # Initially disabled
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Go to the next question',
    icon='arrow-right'
)

# Output widgets for different sections
question_display_area = widgets.Output() # For displaying the question and radio buttons
feedback_display_area = widgets.Output() # For displaying correct/incorrect feedback and final score
score_display_area = widgets.Output() # For displaying the current score

# Global variables for quiz state
current_correct_answer = None
current_question_text = None
current_question_index = 0 # Track current question by index
score = 0
question_attempted = {q: False for q in quiz_data.keys()}
question_answered_correctly_status = {q: False for q in quiz_data.keys()}


# Function to update the score display
def update_score_display():
    with score_display_area:
        score_display_area.clear_output()
        display(HTML(f"<b>Score: {score} / {len(quiz_data)}</b>"))


# Function to render a question by its index
def render_question(index):
    global current_question_text, current_correct_answer, current_question_index
    current_question_index = index

    with question_display_area:
        question_display_area.clear_output() # Clear previous question and radio buttons
        if 0 <= current_question_index < len(question_keys):
            current_question_text = question_keys[current_question_index]
            data = quiz_data[current_question_text]
            options = data['options']
            current_correct_answer = data['answer'] # Store the correct answer globally

            # Create radio buttons for the options of the selected question
            radio_buttons = widgets.RadioButtons(
                options=options,
                description='Choose your answer:',
                disabled=False
            )
            # Attach the answer selection observer to these radio buttons
            radio_buttons.observe(on_answer_selected, names='value')

            # Display the question and the radio buttons
            display(HTML(f"<h3>Selected Question: <span style='color: #333;'>{current_question_text}</span></h3>"))
            display(radio_buttons)

            # Update next button state and description
            next_button.disabled = not question_attempted[current_question_text] # Enable if already attempted
            if current_question_index == len(question_keys) - 1: # If it's the last question
                if all(question_attempted.values()):
                    next_button.description = 'Finish Quiz'
                else:
                    next_button.description = 'End Quiz' # User can end early if not all attempted
            else:
                next_button.description = 'Next Question'

        else:
            display(HTML("<p style='color: red;'>No more questions or invalid question index.</p>"))
            next_button.disabled = True # Disable next button if no more questions

    with feedback_display_area:
        feedback_display_area.clear_output() # Clear feedback when a new question is selected

    update_score_display()


# Function to handle radio button selection (when an answer is chosen)
def on_answer_selected(change):
    global score, question_attempted, question_answered_correctly_status, current_question_text

    selected_option = change['new']

    with feedback_display_area:
        feedback_display_area.clear_output() # Clear previous feedback for this question

        # Mark this question as attempted
        if not question_attempted[current_question_text]:
            question_attempted[current_question_text] = True

        if selected_option == current_correct_answer:
            # If it's the first time this question is answered correctly
            if not question_answered_correctly_status[current_question_text]:
                score += 1
                question_answered_correctly_status[current_question_text] = True
            display(HTML(f"""
                <div style="border: 1px solid #d4edda; padding: 10px; margin-top: 10px; border-radius: 5px; background-color: #d4edda; color: #155724;">
                    <h3>Correct!</h3>
                    <p>The correct answer is: <strong>{current_correct_answer}</strong></p>
                </div>
            """))
        else:
            # If it's incorrect, and it was previously correct, decrement score
            if question_answered_correctly_status[current_question_text]:
                score -= 1 # This allows changing a correct answer to incorrect
                question_answered_correctly_status[current_question_text] = False
            display(HTML(f"""
                <div style="border: 1px solid #f8d7da; padding: 10px; margin-top: 10px; border-radius: 5px; background-color: #f8d7da; color: #721c24;">
                    <h3>Incorrect.</h3>
                    <p>The correct answer is: <strong>{current_correct_answer}</strong></p>
                </div>
            """))

        # Enable the next button as an answer has been attempted
        next_button.disabled = False
        if current_question_index == len(question_keys) - 1: # If it's the last question
            if all(question_attempted.values()):
                next_button.description = 'Finish Quiz'
            else:
                next_button.description = 'End Quiz'
        else:
            next_button.description = 'Next Question'

        # Check if all questions have been attempted after this selection
        if all(question_attempted.values()):
            display(HTML(f"""
                <div style="border: 1px solid #cce5ff; padding: 10px; margin-top: 15px; border-radius: 5px; background-color: #cce5ff; color: #004085;">
                    <h2>Quiz Finished! Your final score is: <strong>{score} / {len(quiz_data)}</strong></h2>
                </div>
            """))
    update_score_display()


# Function to handle next button click
def on_next_button_click(button):
    global current_question_index
    if current_question_index < len(question_keys) - 1: # Not the last question
        current_question_index += 1
        render_question(current_question_index)
        next_button.disabled = True # Disable until next question is attempted
    elif current_question_index == len(question_keys) - 1: # Last question
        # If all questions are attempted, score has already been shown by on_answer_selected.
        # If not all attempted, but 'End Quiz' is clicked, show final score now.
        if not all(question_attempted.values()): # Only show if not already shown
            with feedback_display_area:
                feedback_display_area.clear_output()
                display(HTML(f"""
                    <div style="border: 1px solid #cce5ff; padding: 10px; margin-top: 15px; border-radius: 5px; background-color: #cce5ff; color: #004085;">
                        <h2>Quiz Finished! Your final score is: <strong>{score} / {len(quiz_data)}</strong></h2>
                    </div>
                """))
        next_button.disabled = True # Disable after finishing/ending quiz
        next_button.description = 'Quiz Ended'

# Attach observers
next_button.on_click(on_next_button_click)

# Arrange the widgets in a VBox
ui = widgets.VBox([
    widgets.HBox([next_button, score_display_area]), # Only next_button and score_display
    question_display_area,
    feedback_display_area
])

# Display the UI
display(ui)

# Render the first question initially
render_question(0)
update_score_display()

This example uses `ipywidgets.Dropdown` where the `options` are the keys from the `quiz_data` dictionary. When you select a question from the dropdown, the `on_question_change` function is triggered, which then looks up the corresponding data (options and answer) in the `quiz_data` dictionary and displays it. You can expand `quiz_data` with more questions and their associated data as needed.